In [1]:
import os
import numpy as np
import pandas as pd

# ============ IMPORT YOUR STRATEGY MODULES ============
# These should be in your project root: trading_webapp/strategies/{carry.py, ewma.py}
from strategies.carry import compute_carry_dataframe
from strategies.ewma  import compute_ewma_dataframe

# ============ CONFIG ============
BASE      = r"C:\Users\loci_\Desktop\trading_webapp\DATA"
IN_DIR    = os.path.join(BASE, "all_input_files")
OUT_DIR   = os.path.join(BASE, "all_output_files")
os.makedirs(OUT_DIR, exist_ok=True)

TRADING_DAYS = 256
CAP = 20.0

# Strategy weights inside each instrument (sum to 1.0)
W_STRAT = {"carry": 0.5, "ewma": 0.5}

# Instrument universe + params
# distance: 1/12 for monthly (AD1), 3/12 for quarterly (RX1)
# ewma_fast -> slow is fast*4; scaler per your mapping
INSTRUMENTS = {
    "RX1": {
        "file": os.path.join(IN_DIR, "RX1_small.csv"),
        "distance": 3/12,
        "ewma_fast": 4,
        "ewma_scaler": 7.5,   # from your table
    },
    "AD1": {
        "file": os.path.join(IN_DIR, "AD1_small.csv"),
        "distance": 1/12,
        "ewma_fast": 4,
        "ewma_scaler": 7.5,
    },
}

# EWMA variance lookback for vol-normalisation inside ewma module (if used there)
EWMA_VOL_LOOKBACK = 36  # decay alpha = 2/(N+1)

# ============ HELPERS ============
def ann_sharpe(x, td=TRADING_DAYS):
    s = pd.Series(x).dropna()
    if len(s) < 3:
        return np.nan
    mu, sd = s.mean(), s.std()
    return np.nan if (sd == 0 or np.isnan(sd)) else (mu / sd) * np.sqrt(td)

def compute_fdm(pnl_a: pd.Series, pnl_b: pd.Series, w_a: float, w_b: float, cap: float = 2.5) -> float:
    mat = pd.concat([pnl_a, pnl_b], axis=1).dropna()
    if len(mat) < 30:  # need enough overlap
        return 1.0
    C = mat.corr().values  # 2x2
    w = np.array([w_a, w_b]).reshape(-1, 1)
    denom = float(w.T @ C @ w)
    if denom <= 0 or np.isnan(denom):
        return 1.0
    fdm = 1.0 / np.sqrt(denom)
    return float(min(fdm, cap))

def ensure_px_col(df):
    # robustly locate the price column
    for c in ["PX_CLOSE_1D", "px_close_1d", "Close", "close"]:
        if c in df.columns:
            return c
    raise KeyError("Price column not found (expected PX_CLOSE_1D / px_close_1d / Close).")

# ============ MAIN ============
def run_all():
    portfolio_raw_series = {}  # per-instrument combined raw pnl for optional portfolio view

    for inst, cfg in INSTRUMENTS.items():
        path = cfg["file"]
        if not os.path.exists(path):
            print(f"❌ Missing file for {inst}: {path}")
            continue

        # ---- Load once
        raw = pd.read_csv(path)
        if "Date" in raw.columns:
            raw["Date"] = pd.to_datetime(raw["Date"], dayfirst=True, errors="coerce")
            raw = raw.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

        # ---- Build CARRY forecast (your exact spec)
        carry_df = compute_carry_dataframe(
            df_in=raw,
            distance=cfg["distance"],
            lookback=36,            # your std-dev decay lookback
            forecast_scalar=30.0,   # carry scaler
            cap=CAP,
        )
        # ensure naming
        if "forecast" in carry_df.columns and "forecast_carry" not in carry_df.columns:
            carry_df = carry_df.rename(columns={"forecast": "forecast_carry"})

        # ---- Build EWMA forecast (fast/slow=fast*4, your scaler table)
        ewma_df = compute_ewma_dataframe(
            df_in=raw,
            fast=cfg["ewma_fast"],
            slow=cfg["ewma_fast"] * 4,
            vol_lookback=EWMA_VOL_LOOKBACK,
            forecast_scalar=cfg["ewma_scaler"],
            cap=CAP,
        )
        if "forecast" in ewma_df.columns and "forecast_ewma" not in ewma_df.columns:
            ewma_df = ewma_df.rename(columns={"forecast": "forecast_ewma"})

        # ---- Merge on Date
        left = carry_df[["Date", "forecast_carry"]].copy()
        right = ewma_df[["Date", "forecast_ewma"]].copy()
        df = pd.merge(left, right, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

        # ---- Get returns (single stream): next-day % return of PX_CLOSE_1D
        px_col = ensure_px_col(raw)
        px = raw.sort_values("Date")[px_col].astype(float)
        ret_pct = px.pct_change().rename("ret_pct")
        # align by Date
        tmp = raw[["Date"]].copy()
        tmp["ret_pct"] = ret_pct.values
        df = pd.merge(df, tmp, on="Date", how="left").sort_values("Date").reset_index(drop=True)

        # ---- Strategy raw PnL series (for FDM)
        pnl_carry = df["forecast_carry"].shift(1) * df["ret_pct"]
        pnl_ewma  = df["forecast_ewma"].shift(1)  * df["ret_pct"]

        # ---- FDM (Carver)
        wc, we = W_STRAT["carry"], W_STRAT["ewma"]
        fdm = compute_fdm(pnl_carry, pnl_ewma, wc, we, cap=2.5)

        # ---- Combined forecast (re-cap)
        f_comb = (wc * df["forecast_carry"] + we * df["forecast_ewma"]) * fdm
        f_comb = f_comb.clip(-CAP, CAP)
        df["forecast_combined"] = f_comb

        # ---- Raw Sharpe prints
        sh_carry = ann_sharpe(pnl_carry)
        sh_ewma  = ann_sharpe(pnl_ewma)
        sh_comb  = ann_sharpe(f_comb.shift(1) * df["ret_pct"])
        print(f"\n[{inst}]  FDM={fdm:.3f}")
        print(f"  Carry raw Sharpe: {sh_carry:.3f}")
        print(f"  EWMA  raw Sharpe: {sh_ewma:.3f}")
        print(f"  COMBINED raw Sharpe: {sh_comb:.3f}")

        # ---- Save combined CSV per instrument
        out = df[["Date", "forecast_carry", "forecast_ewma", "forecast_combined"]].copy()
        out_path = os.path.join(OUT_DIR, f"{inst}_COMBINED.csv")
        out.to_csv(out_path, index=False)
        print(f"  Saved: {out_path}")

        # Keep for optional portfolio view
        portfolio_raw_series[inst] = (f_comb.shift(1) * df["ret_pct"]).rename(inst)

    # ---- Optional: simple 50/50 portfolio over overlapping dates
    if len(portfolio_raw_series) >= 2:
        port = pd.concat(portfolio_raw_series.values(), axis=1).dropna()
        weights = {k: 1.0 / len(portfolio_raw_series) for k in portfolio_raw_series}
        port_ret = sum(weights[k] * port[k] for k in port.columns)
        print(f"\n[PORTFOLIO ({' | '.join(port.columns)})] raw Sharpe: {ann_sharpe(port_ret):.3f}")

if __name__ == "__main__":
    run_all()


ModuleNotFoundError: No module named 'strategies'